### 1. 팀 연구 질문 및 추정 목표
- **노출 (X)**: 규칙적 운동 수행 여부 (`exercise`, 1: 비운동군, 2: 규칙적 운동군)
- **결과 (Y)**: 고혈압 유병 여부 (`htn`, 0: 정상, 1: 고혈압)
- **추정 목표**: 규칙적 운동이 고혈압 유병에 미치는 **총효과(Total Effect)** 추정

---





### 2. DAG 작성
age -> exercise
age -> htn
sex -> exercise
sex -> htn
income -> exercise
income -> htn
smk -> htn
drink -> htn
exercise -> bmi
bmi -> htn
exercise -> htn

### 3. 변수 분류표
| 변수명 | 유형 | 근거 |
| :--- | :--- | :--- |
| `age` (연령) | 교란변수 | 연령 증가는 신체활동 능력과 고혈압 발병 위험의 공통 원인이다. |
| `sex` (성별) | 교란변수 | 성별에 따라 운동 실천율과 고혈압 위험이 다르다. |
| `income` (소득) | 교란변수 | 소득 수준은 운동 여가시간 확보 및 보건의료 접근성에 영향을 준다. |
| `smk` (현재흡연) | 공통원인/교란 | 불건강한 생활습관의 동반 가능성 및 혈압에 영향을 미친다. |
| `drink` (음주) | 공통원인/교란 | 고위험 음주는 운동 실천 및 혈압 상승과 관련되어 있다. |
| `bmi` (체질량지수) | 매개변수 | 규칙적 운동이 체중/체지방 변화를 유발하고, 이 변화가 혈압에 영향을 주는 인과 경로상에 위치한다. |

- **최소충분보정집합**: {`age`, `sex`, `income`, `smk`, `drink`}
- **선정 이유**: 노출과 결과 사이의 백도어 경로를 차단하기 위해 주요 교란변수들을 보정하되, 매개변수인 `bmi`는 과보정 편향(Overcontrolling Bias)을 방지하기 위해 주 모형에서 제외함.

In [15]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. 데이터 불러오기
df = pd.read_csv('/content/knhanes_like_week4 (4).csv')

# 2. 가중치 정규화 (필수!)
df['w_norm'] = df['wt'] / df['wt'].mean()

# 3. 규칙적 운동 변수 이분형 변환 (2: 규칙적 운동군 -> 1, 1: 비운동군 -> 0)
df['exercise_bin'] = np.where(df['exercise'] == 2, 1, 0)

# 4. 결측치 제거
vars_needed = ['exercise_bin', 'htn', 'age', 'sex', 'income', 'smk', 'drink', 'bmi', 'w_norm']
df_clean = df.dropna(subset=vars_needed).copy()

# 5. 회귀분석 실행 (HC1 = 샌드위치 표준오차)
# (1) B-2 무보정 모형
m_unadj = smf.glm('htn ~ exercise_bin', data=df_clean, family=sm.families.Binomial(), freq_weights=df_clean['w_norm']).fit(cov_type='HC1')

# (2) B-2 보정 모형 (최소충분보정집합)
m_adj = smf.glm('htn ~ exercise_bin + age + sex + income + smk + drink', data=df_clean, family=sm.families.Binomial(), freq_weights=df_clean['w_norm']).fit(cov_type='HC1')

# (3) B-3 과보정 모형 (매개변수 bmi 일부러 포함)
m_over = smf.glm('htn ~ exercise_bin + age + sex + income + smk + drink + bmi', data=df_clean, family=sm.families.Binomial(), freq_weights=df_clean['w_norm']).fit(cov_type='HC1')

# 6. OR과 95% 신뢰구간 출력 함수
def print_or(model, name):
    coef = model.params['exercise_bin']
    se = model.bse['exercise_bin']
    or_val = np.exp(coef)
    ci_low = np.exp(coef - 1.96 * se)
    ci_high = np.exp(coef + 1.96 * se)
    print(f"{name} OR: {or_val:.2f} ({ci_low:.2f} - {ci_high:.2f})")

print_or(m_unadj, "1. Unadjusted")
print_or(m_adj, "2. Adjusted")
print_or(m_over, "3. Overcontrolled")

1. Unadjusted OR: 0.92 (0.81 - 1.05)
2. Adjusted OR: 0.90 (0.78 - 1.04)
3. Overcontrolled OR: 0.90 (0.77 - 1.04)


### 4. 과보정 시연 및 인과구조 설명 (B-3)
- **무보정 모형 (Unadjusted) OR**: 0.92 (0.81 - 1.05)
- **보정 모형 (Adjusted) OR**: 0.90 (0.78 - 1.04)
- **과보정 모형 (Overcontrolled) OR**: 0.90 (0.77 - 1.04)

- **인과구조 설명**:
  최소충분보정집합(연령, 성별, 소득, 흡연, 음주)을 보정한 모형에서 규칙적 운동군의 고혈압 오즈비는 0.90 (0.78 - 1.04)입니다. 여기에 매개변수인 체질량지수(`bmi`)를 추가한 과보정 모형에서도 오즈비가 0.90 (0.77 - 1.04)으로 변화가 거의 관찰되지 않았습니다. 이는 본 데이터셋에서 규칙적 운동이 고혈압에 미치는 영향 중 BMI를 거치는 매개 경로의 영향이 미미하거나, 이미 보정된 성별·연령 등의 주요 교란변수에 의해 통제되었기 때문입니다. 이처럼 과보정 모형에서 추정치 변화가 거의 없는 사실 자체를 인과구조적으로 확인하였습니다.

---


### 5. Table 2 및 Methods (B-2)

| Variable | Unadjusted OR (95% CI) | Adjusted OR (95% CI) |
| :--- | :--- | :--- |
| **Regular Exercise (`exercise`)** | | |
| Non-exercise (Ref) | 1.00 | 1.00 |
| Regular exercise | 0.92 (0.81 - 1.05) | 0.90 (0.78 - 1.04) |

*Footnote:*
1. Complex survey logistic regression model using normalized sampling weights (`wt / wt.mean()`) with robust sandwich standard errors (HC1).
2. Adjusted for age, sex, income level, current smoking, and high-risk drinking.
3. **Table 2 Fallacy Warning:** 보정변수(Age, Sex, Income, Smoking, Drinking)의 계수 및 오즈비는 인과적으로 해석할 수 없으며, 관심 노출변수(Regular exercise)의 효과를 통제하기 위한 목적으로만 사용되었습니다.

**Methods**
본 연구에서는 국민건강영양조사 유사 데이터를 활용하여 규칙적 운동 수행 여부가 고혈압 유병에 미치는 인과적 총효과를 추정하고자 복합표본 로지스틱 회귀분석을 실시하였다. 표본의 대표성을 반영하기 위해 정규화된 가중치(`wt / wt.mean()`)를 적용하였으며, 설계 기반 샌드위치 표준오차를 산출하여 95% 신뢰구간을 계산하였다. DAG 분석을 통해 도출된 최소충분보정집합에 따라 연령, 성별, 소득수준, 현재흡연율, 고위험음주율을 보정변수로 포함하였다. 매개변수인 체질량지수(BMI)는 과보정 편향을 방지하기 위해 주 모형에서 제외하였다.